# Interactive Entity Mapping for Experiments

This notebook demonstrates the **easiest way** to set up entity mapping for your evaluation experiments.

The `EntityMappingHelper` class provides an intuitive workflow:
1. ✅ Auto-detects entities from your model and dataset
2. ✅ Suggests intelligent mappings
3. ✅ Lets you review and modify mappings interactively
4. ✅ Requires you to handle unmapped entities (map or exclude)
5. ✅ Provides clean configuration for your experiment

In [1]:
from presidio_evaluator import InputSample
from presidio_evaluator.entity_mapping import EntityMappingHelper
from presidio_analyzer import AnalyzerEngine


## Step 1: Load Your Dataset and Model

In [2]:
# Load dataset
dataset = InputSample.read_dataset_json("../data/synth_dataset_v2.json")
print(f"Loaded {len(dataset)} samples")

# Initialize your model
analyzer = AnalyzerEngine()
print("Analyzer initialized")

tokenizing input:   0%|          | 1/1500 [00:00<04:02,  6.19it/s]

loading model en_core_web_sm


tokenizing input: 100%|██████████| 1500/1500 [00:04<00:00, 337.09it/s]


Loaded 1500 samples
Analyzer initialized


## Step 2: Review the Suggested Mapping

The helper automatically:
- Detects all entities in your dataset (with counts)
- Detects all entities supported by your model
- Generates suggested mappings using intelligent strategies
- Show mapped and unmapped entities

In [3]:
# Create the helper - it does all the detection automatically
helper = EntityMappingHelper(
    dataset=dataset,
    model=analyzer,
    language="en"
)

helper.review_mapping()

Dataset Entity,→ Model Entity,Samples,Confidence
⚠️ AGE,NOT MAPPED,74,0.00
⚠️ ORGANIZATION,NOT MAPPED,199,0.00
⚠️ TITLE,NOT MAPPED,92,0.00
✓ CREDIT_CARD,CREDIT_CARD,136,1.00
✓ DATE_TIME,DATE_TIME,119,1.00
✓ DOMAIN_NAME,URL,37,1.00
✓ EMAIL_ADDRESS,EMAIL_ADDRESS,49,1.00
✓ GPE,LOCATION,325,1.00
✓ IBAN_CODE,IBAN_CODE,21,1.00
✓ IP_ADDRESS,IP_ADDRESS,14,1.00


## Step 4: Handle Unmapped Entities

**Important**: If there are unmapped entities, you must take action before proceeding.

### Option A: Map to a Model Entity
If the entity should be evaluated, map it to a model entity.

### Option B: Map to None
If the entity should stay in the dataset but the model doesn't support it, map to `None`.
The model will be penalized for not detecting these (counted as False Negatives).

### Option C: Exclude from Evaluation
If you don't want to evaluate this entity at all, exclude it from the dataset.

In [4]:
# Option A: Map dataset entities to model entities
helper.set_mapping("AGE", "DATE_TIME")
helper.set_mapping("TITLE", "PERSON")

# Option B: Map to None (keep in dataset, penalize model for not detecting)
helper.set_mapping("GPE", None)
helper.set_mapping("ORGANIZATION", None)


helper.review_mapping()

✓ Mapping set: AGE → DATE_TIME
   (2 entities still unmapped)
✓ Mapping set: TITLE → PERSON
   (1 entities still unmapped)
✓ Mapping set: GPE → None
   (1 entities still unmapped)
✓ Mapping set: ORGANIZATION → None


Dataset Entity,→ Model Entity,Samples,Confidence
○ GPE,None,325,None ✎
○ ORGANIZATION,None,199,None ✎
✓ CREDIT_CARD,CREDIT_CARD,136,1.00
✓ DATE_TIME,DATE_TIME,119,1.00
✓ DOMAIN_NAME,URL,37,1.00
✓ EMAIL_ADDRESS,EMAIL_ADDRESS,49,1.00
✓ IBAN_CODE,IBAN_CODE,21,1.00
✓ IP_ADDRESS,IP_ADDRESS,14,1.00
✓ PERSON,PERSON,637,1.00
✓ PHONE_NUMBER,PHONE_NUMBER,64,1.00


### Additional Examples

You can also exclude entities if you don't want them in the evaluation at all:

In [5]:
# Example: Exclude dataset entities you don't care about
# helper.exclude_dataset_entities(["INTERNAL_CODE", "SYSTEM_ID"])

## Step 5: Fine-tune Your Mapping (Optional)

### Modify Existing Mappings

In [6]:
# Change a mapping if you disagree with the suggestion
# helper.set_mapping("FIRST_NAME", "PERSON")
# helper.set_mapping("EMAIL", "EMAIL_ADDRESS")

## Step 6: Review Again

In [7]:
# Check your changes
helper.review_mapping(format="compact")

# Or get a quick summary
helper.summary()


MAPPING: 15 mapped, 0 unmapped | Dataset: 17 active, 0 excluded | Model: 18 entities

✅ All entities mapped! Ready to evaluate.
  💡 Entities mapped to None will be counted as False Negatives.

🔧 Manual (4): AGE→DATE_TIME, GPE→None, ORGANIZATION→None, TITLE→PERSON

Mapping Summary:
  • 15 entities mapped
  • 0 entities unmapped
  • 0 dataset entities excluded
  • 0 model entities excluded
  • 4 manual mappings set

✓ Ready to run experiment!


## Step 7: Get Final Configuration

Once all entities are mapped (no unmapped entities remaining), you can get the final configuration:

In [8]:
# Get the final mapping and entities
mapping = helper.get_mapping()
entities_to_keep = helper.get_model_entities_to_use()
filtered_dataset = helper.get_filtered_dataset()

print(f"\nFinal mapping: {len(mapping)} entries")
print(f"Model entities to use: {entities_to_keep}")
print(f"Filtered dataset: {len(filtered_dataset)}/{len(dataset)} samples")


Final mapping: 15 entries
Model entities to use: ['CREDIT_CARD', 'DATE_TIME', 'EMAIL_ADDRESS', 'IBAN_CODE', 'IP_ADDRESS', 'LOCATION', 'NRP', 'PERSON', 'PHONE_NUMBER', 'URL', 'US_DRIVER_LICENSE', 'US_SSN']
Filtered dataset: 1387/1500 samples


## Step 8: Use in Your Experiment

Now you're ready to run your evaluation!

In [9]:
from presidio_evaluator.models import PresidioAnalyzerWrapper
from presidio_evaluator.evaluation import SpanEvaluator

# Create model wrapper
model = PresidioAnalyzerWrapper(
    analyzer_engine=analyzer,
    entities_to_keep=entities_to_keep
)

# Create evaluator with entity mapping
evaluator = SpanEvaluator(
    model=model,
    entity_mapping=mapping  # Entity mapping is now passed to evaluator!
)

# Run evaluation
print("\n✓ Ready to evaluate!")
# results = evaluator.evaluate_all(filtered_dataset)
# print(results)

TypeError: PresidioAnalyzerWrapper.__init__() got an unexpected keyword argument 'entity_mapper'. Did you mean 'entity_mapping'?

## Tips

1. **Start simple**: Review the suggested mapping first
2. **Handle unmapped entities**: You MUST map or exclude them before proceeding
   - Map to a model entity: Entity will be evaluated normally
   - Map to `None`: Entity stays in dataset, model penalized for not detecting (FN)
   - Exclude: Entity removed from dataset and evaluation
3. **Use summary()**: Quick way to check if you're ready
4. **Iterative refinement**: Make changes, review, repeat
5. **Save time**: The helper auto-regenerates after each change

## Error Handling

The helper prevents common mistakes:

In [ ]:
# Trying to get mapping with unmapped entities raises an error
try:
    mapping = helper.get_mapping()
    print(" ✓ Mapping is complete, ready to evaluate!")
except ValueError as e:
    print(f"Error: {e}")
    print("\nAction: Map or exclude unmapped entities first")
    print("  - Map to model entity: helper.set_mapping('ENTITY', 'TARGET')")
    print("  - Map to None: helper.set_mapping('ENTITY', None)")
    print("  - Exclude: helper.exclude_dataset_entities(['ENTITY'])")
